In [1]:
#Imports 

import os, shutil, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from schema import Schema, And, Use, Optional
import shap
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_curve, auc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mplhep as hep
import plot.style as style
import collections
import keras
from keras.layers import BatchNormalization,  Activation, AveragePooling1D, Dense, Conv1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcParams.update({'figure.max_open_warning': 0})
setattr(collections, "MutableMapping", collections.abc.MutableMapping)
np.bool = np.bool_
num_threads = 24
os.environ["OMP_NUM_THREADS"] = str(num_threads)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(num_threads) 
os.environ["TF_NUM_INTEROP_THREADS"] = str(num_threads)

tf.config.threading.set_inter_op_parallelism_threads(
    num_threads
)
tf.config.threading.set_intra_op_parallelism_threads(
    num_threads
)
tf.keras.utils.set_random_seed(420) # not a special number 




I0000 00:00:1790774938.464477  831616 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/software/bm24156/miniconda3/envs/tagger/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#functions for plotting

def loss_history(plot_dir, history):
    """
    plots loss history of the model and saves in output directory 
    """
    for metric in [
                
                "loss",
                # "prune_low_magnitude_nll_output_loss",
                ]:

        fig,ax = plt.subplots(1,1,figsize=style.FIGURE_SIZE)
        hep.cms.label(llabel=style.CMSHEADER_LEFT,rlabel=style.CMSHEADER_RIGHT,ax=ax, fontsize=style.CMSHEADER_SIZE)
        ax.plot(history.history[metric], label='Train Loss', linewidth=style.LINEWIDTH)
        ax.plot(history.history['val_'+metric], label='Validation Loss',linewidth=style.LINEWIDTH)
        ax.grid(True)
        # ax.set_ylabel('Loss')
        ax.set_ylabel('Loss '+metric)
        ax.set_xlabel('Epoch')
        ax.legend(loc='upper right')

        save_path = os.path.join(plot_dir, "loss_"+metric+"_history")
        plt.savefig(f"{save_path}.png", bbox_inches='tight')
        plt.savefig(f"{save_path}.pdf", bbox_inches='tight')

        fig.clf()
def ROC(y_pred, y_test, class_labels, plot_dir, ROC_dict, suffix=None):
    """
        plots ROC curves for all different classes and saves in output directory 
    """
    # Create a colormap for unique colors
    colormap = matplotlib.colormaps['Set1'].resampled(len(class_labels))  # Use 'tab10' with enough colors

    # Create a plot for ROC curves
    fig,ax = plt.subplots(1,1,figsize=style.FIGURE_SIZE)
    hep.cms.label(llabel=style.CMSHEADER_LEFT,rlabel=style.CMSHEADER_RIGHT,ax=ax, fontsize=style.CMSHEADER_SIZE)
    for i, class_label in enumerate(class_labels):

        # Get true labels and predicted probabilities for the current class
        y_true = y_test[:, i]  # Extract the one-hot column for the current class
        y_score = y_pred[:, i] # Predicted probabilities for the current class

        # Compute FPR, TPR, and AUC
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        ROC_dict[class_label] = roc_auc
        # Plot the ROC curve for the current class
        ax.plot(tpr, fpr, label=f'{style.CLASS_LABEL_STYLE[class_label]} (AUC = {roc_auc:.2f})',
                 color=colormap(i), linewidth=style.LINEWIDTH)

    # Plot formatting
    ax.grid(True)
    ax.set_ylabel('Mistag Rate')
    ax.set_xlabel('Signal Efficiency')

    auc_list = [value for key,value in ROC_dict.items()]
    handles, labels = plt.gca().get_legend_handles_labels()
    order = np.argsort(auc_list)
    ax.legend([handles[idx] for idx in order],[labels[idx] for idx in order],loc='upper left',ncol=2,fontsize=style.SMALL_SIZE-3)

    ax.set_yscale('log')
    ax.set_ylim([1e-3, 1.1])

    # Save the plot
    save_path = os.path.join(plot_dir, "basic_ROC" + (f"_{suffix}" if suffix is not None else ""))
    plt.savefig(f"{save_path}.pdf", bbox_inches='tight')
    plt.savefig(f"{save_path}.png", bbox_inches='tight')
    plt.close()

    return ROC_dict
def confusion(y_pred, y_test, class_labels, plot_dir):
    """
    plots confusion matrix and saves in output directory
    """
    
    cm = confusion_matrix(
        np.argmax(y_test, axis=1),
        np.argmax(y_pred, axis=1),
        normalize="true",
        )
    cm = np.round(cm, 3)
    class_labels = {v: k for k, v in class_labels.items()}
    labels = [style.CLASS_LABEL_STYLE[class_labels[i]] for i in range(y_test.shape[1])]

    # Create a plot of the confusion matrix
    fig, ax = plt.subplots(1, 1, figsize=style.FIGURE_SIZE)
    hep.cms.label(llabel=style.CMSHEADER_LEFT, rlabel=style.CMSHEADER_RIGHT, fontsize=style.CMSHEADER_SIZE)
    matrix_display = ConfusionMatrixDisplay(cm, display_labels=labels)

    matrix_display.plot(ax=ax)
    matrix_display.im_.set_clim(0, 1)

    # Remove default the colorbar
    matrix_display.im_.colorbar.remove()

    # Adjust colorbar height to match the plot
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.5)
    plt.colorbar(matrix_display.im_, cax=cax)

    # Save the plot
    plt.savefig(os.path.join(plot_dir, f"confusion_matrix.png"), bbox_inches='tight')
    plt.savefig(os.path.join(plot_dir, f"confusion_matrix.pdf"), bbox_inches='tight')

def shapPlot(shap_values, feature_names, class_names):
    """
    Calculates shap values for all classes
    """
    fig,ax = plt.subplots(1,1,figsize=style.FIGURE_SIZE)
    feature_order = np.argsort(np.sum(np.mean(np.abs(shap_values), axis=1), axis=0))
    num_features = (shap_values[0].shape[1])
    feature_inds = feature_order
    y_pos = np.arange(len(feature_inds))
    left_pos = np.zeros(len(feature_inds))

    axis_color="#333333"
    class_inds = np.argsort([-np.abs(shap_values[i]).mean() for i in range(len(shap_values))])
    colormap = matplotlib.colormaps['Set1'].resampled(len(class_names))  # Use 'tab10' with enough colors

    for i, ind in enumerate(class_inds):
        global_shap_values = np.abs(shap_values[ind]).mean(0)
        label = style.CLASS_LABEL_STYLE[class_names[ind]]
        ax.barh(y_pos, global_shap_values[feature_inds], 0.7, left=left_pos, align='center',label=label,color=colormap(class_inds[i]))
        left_pos += global_shap_values[feature_inds]

    #ax.set_yticklabels([style.INPUT_FEATURE_STYLE[feature_names[i]] for i in feature_inds])
    ax.legend(loc='lower right',fontsize=30)

    ax.xaxis.set_ticks_position('bottom')
    ax.xaxis.set_ticks_position('none')
    ax.yaxis.set_ticks_position('none')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(color=axis_color, labelcolor=axis_color)
    ax.set_yticks(range(len(feature_order)), [style.INPUT_FEATURE_STYLE[feature_names[i]] for i in feature_order],fontsize=30)
    ax.set_xlabel("mean (Shapley value) - (average impact on model output magnitude)",fontsize=30)
    plt.tight_layout()

def plot_shaply(model, X_test, class_labels, input_vars, plot_dir):
    """
    plots shap values for all classes and saves it in output directory 
    """
    labels = list(class_labels.keys())
    model2 = tf.keras.Model(model.input, model.output)
   

    for explainer, name  in [(shap.GradientExplainer(model2, X_test[:1000]), "GradientExplainer"), ]:
        print("... {0}: explainer.shap_values(X)".format(name))
        shap_values = explainer.shap_values(X_test[:1000])
        new = np.sum(shap_values, axis = 1)
        print("... shap summary_plot classification")
        plt.clf()
        new = np.transpose(new, (2,0,1))
        shapPlot(new, input_vars, labels)
        plt.savefig(plot_dir+"/shap_summary_class.pdf",bbox_inches='tight')
        plt.savefig(plot_dir+"/shap_summary_class.png",bbox_inches='tight')

In [3]:
# Load training data 
X_train = np.load('training_data_CMS/X_train.npy')
y_train = np.load('training_data_CMS/y_train.npy')


In [ ]:
# define model
# DeepSet architecture 
def jet_tagger(input_shape,output_shape):
    L, C = input_shape
    inputs = keras.layers.Input(shape=input_shape, name='model_input')
    main = BatchNormalization(name='norm_input')(inputs)
    main = Conv1D(filters=10, kernel_size=1, name='Conv1D_1', activation='relu')(main)
    main = Conv1D(filters=10, kernel_size=1, name='Conv1D_2', activation='relu')(main)
    main = AveragePooling1D(L,name='avgpool')(main)
    main = Flatten()(main)
    jet_id = Dense(32, name='Dense_1' ,activation='relu')(main)
    jet_id = Dense(16, name='Dense_2' ,activation='relu')(jet_id)
    jet_id = Dense(8, name='Dense_3' )(jet_id)
    jet_id = Activation('softmax', name='jet_id_output')(jet_id)
    jet_model = keras.Model(inputs = [inputs], outputs = [jet_id])
    return jet_model

In [5]:
input_shape = X_train.shape[1:] # First dimension is batch size
output_shape = y_train.shape[1:]
model = jet_tagger(input_shape, output_shape) 
callbacks = [
            EarlyStopping(monitor='val_loss', patience=10),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=0.00001,
            ),
        ]

model.compile(
            optimizer='adam',
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics={'jet_id_output': 'categorical_accuracy'}
        )

E0000 00:00:1790774973.103759  831616 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_SYSTEM_DRIVER_MISMATCH: system has unsupported display driver / cuda driver combination
I0000 00:00:1790774973.103813  831616 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1790774973.103817  831616 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: hm01.dice.priv
I0000 00:00:1790774973.103822  831616 cuda_diagnostics.cc:183] hostname: hm01.dice.priv
I0000 00:00:1790774973.104007  831616 cuda_diagnostics.cc:190] libcuda reported version is: 610.57.4
I0000 00:00:1790774973.104024  831616 cuda_diagnostics.cc:194] kernel reported version is: 610.43.2
E0000 00:00:1790774973.104026  831616 cuda_diagnostics.cc:287] kernel version 610.43.2 does not match DSO version 610.57.4 -- cannot find workin

In [6]:
# train
BATCH_SIZE = 512
EPOCHS = 10
VALIDATION_SPLIT = 0.20
history = model.fit( X_train,y_train,
                            epochs=EPOCHS,
                            batch_size=BATCH_SIZE,
                            verbose=2,
                            validation_split=VALIDATION_SPLIT,
                            callbacks = [callbacks],
                            shuffle=True)

Epoch 1/10
22937/22937 - 102s - 4ms/step - categorical_accuracy: 0.5375 - loss: 1.1897 - val_categorical_accuracy: 0.5555 - val_loss: 1.1455 - learning_rate: 0.0010
Epoch 2/10
22937/22937 - 98s - 4ms/step - categorical_accuracy: 0.5575 - loss: 1.1387 - val_categorical_accuracy: 0.5604 - val_loss: 1.1311 - learning_rate: 0.0010
Epoch 3/10
22937/22937 - 98s - 4ms/step - categorical_accuracy: 0.5602 - loss: 1.1310 - val_categorical_accuracy: 0.5622 - val_loss: 1.1276 - learning_rate: 0.0010
Epoch 4/10
22937/22937 - 98s - 4ms/step - categorical_accuracy: 0.5615 - loss: 1.1282 - val_categorical_accuracy: 0.5629 - val_loss: 1.1257 - learning_rate: 0.0010
Epoch 5/10
22937/22937 - 98s - 4ms/step - categorical_accuracy: 0.5622 - loss: 1.1267 - val_categorical_accuracy: 0.5633 - val_loss: 1.1249 - learning_rate: 0.0010
Epoch 6/10
22937/22937 - 100s - 4ms/step - categorical_accuracy: 0.5627 - loss: 1.1256 - val_categorical_accuracy: 0.5635 - val_loss: 1.1240 - learning_rate: 0.0010
Epoch 7/10
229

In [7]:
# save
export_path = os.path.join("output", "model/saved_model.h5")
model.save(export_path)

In [8]:
#plot using the functions above 

plot_dir = os.path.join("output", "plots/training")
os.makedirs(plot_dir, exist_ok=True)
#Load the metadata for class_label
with open(f"training_data_CMS/class_label.json", 'r') as file: class_labels = json.load(file)
with open(f"training_data_CMS/input_vars.json", 'r') as file: input_vars = json.load(file)

ROC_dict = {class_label : 0 for class_label in class_labels}

#load test data
X_test = np.load(f"training_data_CMS/X_test.npy")
y_test = np.load(f"training_data_CMS/y_test.npy")

#load saved model
model = load_model(f"output/model/saved_model.h5")
model_outputs = model.predict(X_test)
y_pred = model_outputs
    
ROC_dict = ROC(y_pred, y_test, class_labels, plot_dir,ROC_dict)
confusion(y_pred, y_test, class_labels, plot_dir)
plot_shaply(model, X_test, class_labels, input_vars, plot_dir)

50971/50971 ━━━━━━━━━━━━━━━━━━━━ 29s 572us/step
... GradientExplainer: explainer.shap_values(X)
... shap summary_plot classification
